# URSim ↔ RTDE smoke test notebook (ur_rtde)

This notebook tests **connectivity**, **state receiving**, and **basic command sending** against a URSim Docker container.

You are on macOS (arm64). The URSim container runs as linux/amd64 and is accessed via forwarded ports.

> Safety: Even in simulation, treat motion commands as potentially unsafe. Start with the **Receive-only** section first.


## 0) URSim Docker command (ports)

For **receive-only** (RTDE state): forward **30004**.

For **control** with `ur_rtde` (move/servo), you typically also need:
- **29999** (Dashboard server)
- **30002** (URScript server)

Run URSim like this (host loopback-only):

```bash
docker stop ursim 2>/dev/null || true
docker rm ursim 2>/dev/null || true

docker run -d --name ursim --platform linux/amd64 \
  -p 127.0.0.1:5900:5900 \
  -p 127.0.0.1:6080:6080 \
  -p 127.0.0.1:30004:30004 \
  -p 127.0.0.1:30002:30002 \
  -p 127.0.0.1:29999:29999 \
  universalrobots/ursim_e-series
```

Polyscope UI:
- http://localhost:6080/vnc.html?host=localhost&port=6080


## 1) Environment sanity checks


In [1]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Machine:", platform.machine())


Python: 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:46:49) [Clang 19.1.7 ]
Platform: macOS-26.3-arm64-arm-64bit
Machine: arm64


## 3) Import ur_rtde modules

`pip list | grep -i rtde` should show `ur_rtde`.

Import names are:
- `rtde_receive`
- `rtde_control`
- (optional) `rtde_io`


In [2]:
import rtde_receive, rtde_control
print("rtde_receive:", rtde_receive.__file__)
print("rtde_control:", rtde_control.__file__)


rtde_receive: /Users/matthiasweiss/miniconda3/envs/mujoco/lib/python3.10/site-packages/rtde_receive.cpython-310-darwin.so
rtde_control: /Users/matthiasweiss/miniconda3/envs/mujoco/lib/python3.10/site-packages/rtde_control.cpython-310-darwin.so


## 4) Receive-only smoke test

This should work with **only** port 30004 forwarded.

It reads:
- joint positions `q`
- TCP pose
- robot mode / safety mode (availability depends on controller version)


In [3]:
HOST = "127.0.0.1"   # or "localhost"

r = rtde_receive.RTDEReceiveInterface(HOST)
print("Connected:", r.isConnected())

print("q  =", r.getActualQ())
print("qd =", r.getActualQd())
print("tcp=", r.getActualTCPPose())

r.disconnect()

Connected: True
q  = [-1.6006999999999998, -1.7271, -2.2029999999999994, -0.8079999999999998, 1.5951, -0.030999999999999694]
qd = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
tcp= [-0.14396865671352163, -0.43562006080319216, 0.2020300254283846, -0.0012213596815925638, 3.1162765284819756, 0.038891915636886834]


## 5) Live receive loop (10 seconds)

If you see intermittent `End of file` errors, it's typically a connection/program state issue.


In [4]:
import time
import rtde_receive

HOST = "127.0.0.1"
DT = 0.005        # 200 Hz
N_UPDATES = 10

r = rtde_receive.RTDEReceiveInterface(HOST)

next_t = time.perf_counter()

try:
    for i in range(N_UPDATES):

        # read robot state
        q = r.getActualQ()
        tcp = r.getActualTCPPose()

        print(f"step {i:02d}  q={['%+.3f'%v for v in q]}  tcp_z={tcp[2]:+.3f}")

        # maintain constant update rate
        next_t += DT
        sleep_time = next_t - time.perf_counter()
        if sleep_time > 0:
            time.sleep(sleep_time)

finally:
    r.disconnect()


step 00  q=['-1.601', '-1.727', '-2.203', '-0.808', '+1.595', '-0.031']  tcp_z=+0.202
step 01  q=['-1.601', '-1.727', '-2.203', '-0.808', '+1.595', '-0.031']  tcp_z=+0.202
step 02  q=['-1.601', '-1.727', '-2.203', '-0.808', '+1.595', '-0.031']  tcp_z=+0.202
step 03  q=['-1.601', '-1.727', '-2.203', '-0.808', '+1.595', '-0.031']  tcp_z=+0.202
step 04  q=['-1.601', '-1.727', '-2.203', '-0.808', '+1.595', '-0.031']  tcp_z=+0.202
step 05  q=['-1.601', '-1.727', '-2.203', '-0.808', '+1.595', '-0.031']  tcp_z=+0.202
step 06  q=['-1.601', '-1.727', '-2.203', '-0.808', '+1.595', '-0.031']  tcp_z=+0.202
step 07  q=['-1.601', '-1.727', '-2.203', '-0.808', '+1.595', '-0.031']  tcp_z=+0.202
step 08  q=['-1.601', '-1.727', '-2.203', '-0.808', '+1.595', '-0.031']  tcp_z=+0.202
step 09  q=['-1.601', '-1.727', '-2.203', '-0.808', '+1.595', '-0.031']  tcp_z=+0.202


## 6) Before sending motion commands in URSim

Open Polyscope (VNC) and ensure:
1. The robot is **Powered on**
2. **Brakes released**
3. No protective stop / safety popup blocking motion

If Dashboard is used, the script can also query/unlock, but in URSim it's often simplest to do it once in the UI.


## 7) Control smoke test: connect + stopScript

This requires **30002** (URScript) and often **29999** (Dashboard) forwarded.


In [5]:
import socket

for port in [30002, 30004, 29999, 50002]:
    try:
        s = socket.create_connection(("127.0.0.1", port), timeout=2)
        print("OK:", port)
        s.close()
    except Exception as e:
        print("FAIL:", port, e)

OK: 30002
OK: 30004
OK: 29999
OK: 50002


Minimal control test (URScript over 30002)

In [6]:
import socket

HOST = "127.0.0.1"
PORT = 50002

script = """
def test():
  movej([0,-1.57,1.57,0,1.57,0], a=0.5, v=0.5)
end
test()
"""

s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.connect((HOST, PORT))
s.send(script.encode('utf-8'))
s.close()

print("URScript sent")


URScript sent


In [7]:
import socket, math

HOST, PORT = "127.0.0.1", 30002

q_home = [0.0, -math.pi/2, math.pi/2, 0.0, math.pi/2, 0.0]

script = f"""
def py_move():
  movej({q_home}, a=0.8, v=0.8)
end
py_move()
"""

with socket.create_connection((HOST, PORT), timeout=2) as s:
    s.sendall(script.encode("utf-8"))
    

In [8]:
import time
import socket
import rtde_receive

HOST = "127.0.0.1"

r = rtde_receive.RTDEReceiveInterface(HOST)

# read state
q = r.getActualQ()
print("current q:", q)

# send URScript command
script = f"""
def policy_move():
  movej({q}, a=0.5, v=0.5)
end
policy_move()
"""

s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.connect((HOST, 30002))
s.send(script.encode())
s.close()

r.disconnect()


current q: [-1.5852032792651167, -1.7255867905457016, -2.1664650666666656, -0.8001775783383609, 1.594864710916167, -0.03069988233723908]


## 8) Control: MoveJ to a safe 'home' pose (example)

Set a joint home for your robot model.
- UR3 has 6 joints.
- Units are **radians**.

Start with small motions in simulation.


In [10]:
import rtde_control

HOST = "127.0.0.1"

flags = rtde_control.RTDEControlInterface.FLAG_UPLOAD_SCRIPT
# optional if you don’t want it to block waiting for Play:
# flags |= rtde_control.RTDEControlInterface.FLAG_NO_WAIT

c = rtde_control.RTDEControlInterface(HOST, -1.0, flags)
print("Connected:", c.isConnected())

RuntimeError: connect: Connection refused [system:61 at /opt/homebrew/Cellar/boost@1.85/1.85.0_3/include/boost/asio/detail/reactive_socket_service.hpp:587:33 in function 'connect']

RTDEReceiveInterface boost system Exception: (system:89) Operation canceled [system:89 at /opt/homebrew/Cellar/boost@1.85/1.85.0_3/include/boost/asio/detail/reactive_socket_recv_op.hpp:133:37 in function 'do_complete']


In [9]:
import math, time

HOST = "127.0.0.1"

# Example "home" joint pose (adjust for your URSim model)
Q_HOME = [0.0, -math.pi/2, math.pi/2, 0.0, math.pi/2, 0.0]

r = rtde_receive.RTDEReceiveInterface(HOST)
c = rtde_control.RTDEControlInterface(HOST)


print("q current:", r.getActualQ())

# Conservative speed/acceleration
speed = 0.5
accel = 0.5

ok = c.moveJ(Q_HOME, speed, accel)
print("moveJ returned:", ok)
time.sleep(0.5)
print("q now:", r.getActualQ())

c.stopScript()
c.disconnect()
r.disconnect()


RuntimeError: connect: Connection refused [system:61 at /opt/homebrew/Cellar/boost@1.85/1.85.0_3/include/boost/asio/detail/reactive_socket_service.hpp:587:33 in function 'connect']

## 9) Servo loop template (for learned policies)

Typical structure:
- Receive state
- Compute action
- Send command (servoJ / speedJ / servoL)
- Sleep to keep a fixed control rate

Note: You must ensure your command choice matches how you trained (e.g., joint targets vs velocities).


In [ ]:
import time, math
from typing import List

HOST = "127.0.0.1"

# --- User parameters ---
CONTROL_HZ = 125
DURATION_S = 5.0

# servoJ parameters (tune carefully)
LOOKAHEAD_TIME = 0.1
GAIN = 300

# Starting target (small perturbation around current q)
# -----------------------

def policy_stub(q: List[float], qd: List[float], tcp: List[float]) -> List[float]:
    # Replace with your learned policy.
    # Here: hold position (no change).
    return q

r = rtde_receive.RTDEReceiveInterface(HOST)
c = rtde_control.RTDEControlInterface(HOST)

dt = 1.0 / CONTROL_HZ
t0 = time.time()

try:
    while time.time() - t0 < DURATION_S:
        q = r.getActualQ()
        qd = r.getActualQd()
        tcp = r.getActualTCPPose()

        q_target = policy_stub(q, qd, tcp)

        # Send a servoJ target
        c.servoJ(q_target, 0.5, 0.5, dt, LOOKAHEAD_TIME, GAIN)

        time.sleep(dt)
finally:
    # Stop controller cleanly
    try:
        c.servoStop()
    except Exception:
        pass
    try:
        c.stopScript()
    except Exception:
        pass
    c.disconnect()
    r.disconnect()

print("Done.")


## 10) Notes for later UR3 integration

When you switch to a real UR3:
- Use the robot's IP instead of 127.0.0.1
- Keep the same port expectations (RTDE 30004, script 30002, dashboard 29999)
- Ensure Remote Control / safety state allows program execution
- Add safety checks in code: speed limits, workspace bounds, emergency stop handling
